# $\pi^0$ Dalitz decays: extract, reweight Geant4 $\to$ EvtGen, and plot

End-to-end in one notebook: **(1)** extract truth $\pi^0\to e^+e^-\gamma$ Dalitz decays from the overlay checkout files (recovering decays where a soft lepton fell below the WC storage threshold, via 4-momentum conservation); **(2)** build a reweighting that corrects Geant4's decay kinematics to EvtGen's; **(3)** make the closure and application plots.

**Why a single weight works.** The decay of a spin-0 particle at rest has only **two dynamical degrees of freedom** — the dilepton invariant mass $m_{ee}$ and the lepton helicity angle $\cos\theta^{*}$, both **invariant under the $\pi^0$ lab boost**. So the entire Geant4-vs-EvtGen difference is captured by
$$w(m_{ee}, \cos\theta^{*}) = \frac{\rho_\mathrm{EvtGen}(m_{ee}, \cos\theta^{*})}{\rho_\mathrm{Geant4}(m_{ee}, \cos\theta^{*})}.$$

**Source = standalone Geant4 (1M, pure model)** from `G4DalitzDecayChannel` (isotropic lepton angle), **target = EvtGen (1M)** (`PI0_DALITZ`, the QED $1+\cos^2\theta^{*}$). The weight is built from these high-stats pure-model samples and applied to the (recovered, unbiased) overlay decays using their truth $(m_{ee}, \cos\theta^{*})$.

In [ ]:
import os
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt

M_PI0 = 134.9768  # MeV
M_E = 0.51099895

DATA = "/nevis/riverside/data/leehagaman/ngem/data_files/"
GEANT4_CSV = DATA + "geant4_pi0_dalitz.csv"   # 1M standalone Geant4 (source, pure model)
EVTGEN_CSV = DATA + "evtgen_pi0_dalitz.csv"   # 1M EvtGen (target)

# overlay checkout files to extract pi0 Dalitz decays from (nu + pi0-enriched NC pi0)
INPUT_FILES = [
    ("nu_overlay",     "checkout_MCC9.10_Run123_v10_04_07_20_BNB_nu_overlay_surprise_reco2_hist_1.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4a_v10_04_07_16_BNB_NCpi0_overlay_surprise_reco2_hist.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4b_v10_04_07_09_BNB_NC_pi0_overlay_surprise_reco2_hist.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_4c.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_4d.root"),
    ("nc_pi0_overlay", "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_5.root"),
]

PLOT_DIR = "../plots/pi0_dalitz"
os.makedirs(PLOT_DIR, exist_ok=True)


def savefig(name):
    plt.savefig(os.path.join(PLOT_DIR, name + ".pdf"))
    plt.savefig(os.path.join(PLOT_DIR, name + ".png"), dpi=200)

In [ ]:
# four-vectors as [E, px, py, pz]
def boost_to_rest(p4, frame_p4):
    p4 = np.asarray(p4, dtype=float)
    frame_p4 = np.asarray(frame_p4, dtype=float)
    E_frame = frame_p4[..., 0]
    beta = frame_p4[..., 1:] / E_frame[..., None]
    b2 = np.sum(beta**2, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - b2)
    pe = p4[..., 0]
    pv = p4[..., 1:]
    bp = np.sum(beta * pv, axis=-1)
    with np.errstate(divide="ignore", invalid="ignore"):
        gamma2 = np.where(b2 > 0, (gamma - 1.0) / b2, 0.0)
    E_prime = gamma * (pe - bp)
    pv_prime = pv + (gamma2 * bp - gamma * pe)[..., None] * beta
    return np.concatenate([E_prime[..., None], pv_prime], axis=-1)


def p3(p4):
    return p4[..., 1:]


def minkowski_mass(p4):
    return np.sqrt(np.clip(p4[..., 0]**2 - np.sum(p4[..., 1:]**2, axis=-1), 0, None))


def cos_between(a3, b3):
    c = np.sum(a3 * b3, axis=-1) / (np.linalg.norm(a3, axis=-1) * np.linalg.norm(b3, axis=-1))
    return np.clip(c, -1.0, 1.0)


def observables(gamma, ep, em):
    """All pi0-rest-frame Dalitz observables from lab four-vectors."""
    pi0 = gamma + ep + em
    g = boost_to_rest(gamma, pi0)
    p = boost_to_rest(ep, pi0)
    m = boost_to_rest(em, pi0)
    ee = p + m
    ep_in_ee = boost_to_rest(p, ee)
    return {
        "E_gamma": g[:, 0], "E_ep": p[:, 0], "E_em": m[:, 0],
        "m_ee": minkowski_mass(ee),
        "opening": np.degrees(np.arccos(cos_between(p3(p), p3(m)))),
        "asym": (p[:, 0] - m[:, 0]) / (p[:, 0] + m[:, 0]),
        "cos_theta_l": cos_between(p3(ep_in_ee), p3(ee)),
    }


def obs_from_csv(path):
    d = pd.read_csv(path)
    return observables(d[["E_g", "px_g", "py_g", "pz_g"]].values,
                       d[["E_ep", "px_ep", "py_ep", "pz_ep"]].values,
                       d[["E_em", "px_em", "py_em", "pz_em"]].values)

## Extract $\pi^0$ Dalitz decays from the overlay (with recovery)

Read the WireCell `T_PFeval` truth particles from the overlay checkout files and pick out the $\pi^0\to e^+e^-\gamma$ decays. A $\pi^0$ with **any** $e^\pm$ direct daughter is treated as Dalitz; if one daughter is missing (WC stores $e^+/e^-$ only above ~10 MeV lab energy), it is **recovered** exactly from 4-momentum conservation. This yields an unbiased overlay sample (the recovered Dalitz fraction matches the PDG branching ratio). `truth_startMomentum` is `[px,py,pz,E]` in GeV; we convert to MeV. Both the recovered and complete-only samples are also saved to `.npz` for other notebooks.

In [ ]:
BRANCHES = ["run", "subrun", "event", "truth_id", "truth_pdg", "truth_mother", "truth_startMomentum"]


def extract_dalitz(filepath):
    """Extract pi0 Dalitz decays from one checkout file.

    Any pi0 with at least one e+/e- direct daughter is a Dalitz decay (other leptonic
    channels are negligible). If one of {gamma, e+, e-} is missing (WC stores e+/e-
    only above ~10 MeV lab energy), it is recovered exactly from 4-momentum
    conservation: missing = pi0 - (sum of present daughters). Complete and recovered
    decays are returned together -- we always use the total identifiable Dalitz sample.

    Returns (rows, n_pi0_total, n_complete, n_recovered).
    """
    rows = {k: [] for k in ["pi0", "gamma", "ep", "em", "run", "subrun", "event"]}
    n_pi0_total = n_complete = n_recovered = 0
    for chunk in uproot.iterate(f"{filepath}:wcpselection/T_PFeval", BRANCHES,
                                step_size="500 MB", library="np"):
        for i in range(len(chunk["truth_pdg"])):
            pdg = np.asarray(chunk["truth_pdg"][i])
            if pdg.size == 0:
                continue
            tid = np.asarray(chunk["truth_id"][i])
            moth = np.asarray(chunk["truth_mother"][i])
            sm = np.asarray(chunk["truth_startMomentum"][i], dtype=float)  # [px,py,pz,E] GeV
            for k in np.where(pdg == 111)[0]:
                n_pi0_total += 1
                d_idx = np.where(moth == tid[k])[0]
                dpdg = pdg[d_idx]
                if not ((11 in dpdg) or (-11 in dpdg)):
                    continue

                def p4_mev(idx):  # [px,py,pz,E] GeV -> [E,px,py,pz] MeV
                    px, py, pz, E = sm[idx]
                    return np.array([E, px, py, pz]) * 1000.0

                def first(particle_pdg):
                    sel = d_idx[dpdg == particle_pdg]
                    return p4_mev(sel[0]) if len(sel) else None

                pi0 = p4_mev(k)
                g, ep, em = first(22), first(-11), first(11)
                if g is not None and ep is not None and em is not None:
                    n_complete += 1
                elif g is not None and ep is not None and em is None:
                    em = pi0 - g - ep; n_recovered += 1
                elif g is not None and em is not None and ep is None:
                    ep = pi0 - g - em; n_recovered += 1
                elif ep is not None and em is not None and g is None:
                    g = pi0 - ep - em; n_recovered += 1
                else:
                    continue

                rows["pi0"].append(pi0); rows["gamma"].append(g); rows["ep"].append(ep); rows["em"].append(em)
                rows["run"].append(int(chunk["run"][i]))
                rows["subrun"].append(int(chunk["subrun"][i]))
                rows["event"].append(int(chunk["event"][i]))
    return rows, n_pi0_total, n_complete, n_recovered


acc = {k: [] for k in ["pi0", "gamma", "ep", "em", "run", "subrun", "event"]}
acc_sample = []
n_pi0_total = n_complete = n_recovered = 0
print(f"{'sample':<16} {'pi0s':>10} {'Dalitz':>8}  file")
for label, fname in INPUT_FILES:
    rows, npi0, ncomp, nrec = extract_dalitz(os.path.join(DATA, fname))
    for key in acc:
        acc[key] += rows[key]
    acc_sample += [label] * len(rows["pi0"])
    n_pi0_total += npi0; n_complete += ncomp; n_recovered += nrec
    print(f"{label:<16} {npi0:>10,} {ncomp + nrec:>8,}  {fname}")

ov = {
    "pi0_p4": np.array(acc["pi0"]), "gamma_p4": np.array(acc["gamma"]),
    "ep_p4": np.array(acc["ep"]), "em_p4": np.array(acc["em"]),
    "sample": np.array(acc_sample),
    "run": np.array(acc["run"]), "subrun": np.array(acc["subrun"]), "event": np.array(acc["event"]),
}
n_dalitz = n_complete + n_recovered
print("-" * 60)
print(f"TOTAL truth pi0s: {n_pi0_total:,}   Dalitz: {n_dalitz:,} "
      f"({n_recovered:,} recovered from a missing lepton)")
print(f"Dalitz fraction: {n_dalitz / n_pi0_total * 100:.3f}%  (PDG BR = 1.174%)")

np.savez("pi0_dalitz_momenta.npz", **ov)

In [ ]:
# source (standalone Geant4, pure model) and target (EvtGen), both 1M
def load_daughters(path):
    d = pd.read_csv(path)
    return (d[["E_g", "px_g", "py_g", "pz_g"]].values,
            d[["E_ep", "px_ep", "py_ep", "pz_ep"]].values,
            d[["E_em", "px_em", "py_em", "pz_em"]].values)


g4_obs = observables(*load_daughters(GEANT4_CSV))
ev_obs = observables(*load_daughters(EVTGEN_CSV))

# overlay observables from the (total) Dalitz decays extracted above
ov_obs = observables(ov["gamma_p4"], ov["ep_p4"], ov["em_p4"])

print(f"source  (standalone Geant4): {len(g4_obs['m_ee']):,}")
print(f"target  (EvtGen):            {len(ev_obs['m_ee']):,}")
print(f"overlay (Dalitz total):      {len(ov_obs['m_ee']):,}")

## Build the 2D weight

$w(m_{ee}, \cos\theta^{*})$ as the ratio of EvtGen to standalone-Geant4 densities, on a grid of quantile-spaced $m_{ee}$ bins $\times$ uniform $\cos\theta^{*}$ bins. With 1M decays on each side the grid can be fine, and a full 2D weight captures any $m_{ee}$–$\cos\theta^{*}$ correlation exactly. Weights are renormalized to $\langle w\rangle = 1$ over the source.

In [ ]:
NM, NC = 25, 20  # m_ee bins, cos_theta* bins

mee_edges = np.quantile(g4_obs["m_ee"], np.linspace(0, 1, NM + 1))
mee_edges[0], mee_edges[-1] = 2 * M_E, M_PI0
cos_edges = np.linspace(-1, 1, NC + 1)

Hg, _, _ = np.histogram2d(g4_obs["m_ee"], g4_obs["cos_theta_l"], bins=[mee_edges, cos_edges])
He, _, _ = np.histogram2d(ev_obs["m_ee"], ev_obs["cos_theta_l"], bins=[mee_edges, cos_edges])
fg, fe = Hg / Hg.sum(), He / He.sum()
W2 = np.divide(fe, fg, out=np.zeros_like(fe), where=fg > 0)   # weight per (m_ee, cos) bin


def weight_of(mee, cos):
    """Per-event 2D weight for given (m_ee, cos theta*), looked up from the grid."""
    i = np.clip(np.digitize(mee, mee_edges) - 1, 0, NM - 1)
    j = np.clip(np.digitize(cos, cos_edges) - 1, 0, NC - 1)
    return W2[i, j]


# source weights and overlay weights, each normalized <w> = 1
w = weight_of(g4_obs["m_ee"], g4_obs["cos_theta_l"])
w *= len(w) / w.sum()
w_ov = weight_of(ov_obs["m_ee"], ov_obs["cos_theta_l"])
w_ov *= len(w_ov) / w_ov.sum()

n_eff = w.sum()**2 / np.sum(w**2)
print(f"min Geant4 events/bin: {int(Hg.min())}")
print(f"weight range: {w.min():.2f} .. {w.max():.2f}")
print(f"Kish effective N: {n_eff:.0f} / {len(w)}  ({n_eff/len(w)*100:.0f}%)")

In [ ]:
# weight map and per-event weight distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
Wm = np.ma.masked_where(Hg == 0, W2)
im = axes[0].pcolormesh(np.arange(NM + 1), cos_edges, Wm.T, cmap="RdBu_r", vmin=0, vmax=2)
axes[0].set_xlabel(r"$m_{ee}$ quantile-bin index (low $\to$ high mass)")
axes[0].set_ylabel(r"$\cos\theta^{*}$")
axes[0].set_title(r"weight $w(m_{ee}, \cos\theta^{*})$ = EvtGen / Geant4")
fig.colorbar(im, ax=axes[0], label="weight")
axes[1].hist(w, bins=50, histtype="step", color="tab:green")
axes[1].axvline(1.0, ls=":", color="gray")
axes[1].set_xlabel("per-event weight")
axes[1].set_ylabel("Geant4 decays")
axes[1].set_title(f"weights ($N_{{eff}}$ = {n_eff:.0f} / {len(w)})")
fig.tight_layout()
savefig("reweight_weight_map")
plt.show()

## Closure: the reweighting variables

The two variables the weight is built from. **Geant4 reweighted** (green dashed) lands exactly on **EvtGen** (red solid) by construction — the weight-closure check. The **overlay raw** (purple) sits on the **Geant4 pure model** (blue, flat in $\cos\theta^{*}$), and the **overlay reweighted** (orange dashed) lands on EvtGen and the QED $\tfrac{3}{8}(1+\cos^2\theta^{*})$ curve. $m_{ee}$ is shown log-log.

In [ ]:
def closure(key, bins, xlabel, logx=False, logy=False, ax=None):
    # solids: the uncorrected model and the target
    ax.hist(g4_obs[key], bins=bins, density=True, histtype="step", lw=1.4,
            color="tab:blue", alpha=0.7, label="Geant4 (new generation)")
    ax.hist(ov_obs[key], bins=bins, density=True, histtype="step", lw=1.4,
            color="tab:purple", label="BNB NC $\pi^0$ overlay")
    ax.hist(ev_obs[key], bins=bins, density=True, histtype="step", lw=1.6,
            color="tab:red", label="EvtGen (target)")
    # dashed, drawn on top: both should land on the EvtGen target
    ax.hist(g4_obs[key], bins=bins, density=True, weights=w, histtype="step", lw=1.8,
            ls="--", color="tab:green", label="Geant4 reweighted")
    ax.hist(ov_obs[key], bins=bins, density=True, weights=w_ov, histtype="step", lw=1.8,
            ls="--", color="tab:orange", label="BNB NC $\pi^0$ overlay reweighted")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("normalized")
    if logx:
        ax.set_xscale("log")
    if logy:
        ax.set_yscale("log")
    ax.legend(fontsize=7)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
closure("cos_theta_l", cos_edges, r"$\cos\theta^{*}$", ax=axes[0])
xx = np.linspace(-1, 1, 200)
axes[0].plot(xx, 3.0 / 8.0 * (1.0 + xx**2), "k:", lw=1.5, label=r"$1+\cos^2\theta^{*}$")
axes[0].legend(fontsize=7)
closure("m_ee", mee_edges, r"$m_{ee}$ [MeV]", logx=True, logy=True, ax=axes[1])
fig.suptitle(r"$\pi^0$ Rest Frame")
fig.tight_layout()
savefig("reweight_closure_key")
plt.show()

### $m_{ee}$ on linear and log-linear scales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
closure("m_ee", mee_edges, r"$m_{ee}$ [MeV]", logx=False, logy=False, ax=axes[0])
axes[0].set_title("linear-linear")
closure("m_ee", mee_edges, r"$m_{ee}$ [MeV]", logx=True, logy=False, ax=axes[1])
axes[1].set_title("log-linear")
fig.suptitle(r"$\pi^0$ Rest Frame")
fig.tight_layout()
savefig("reweight_closure_mee_scales")
plt.show()

## Closure 2: other $\pi^0$-rest-frame observables

Not in the weight, but functions of $(m_{ee}, \cos\theta^{*})$, so they should also close onto EvtGen.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
closure("asym", np.linspace(-1, 1, 60), r"energy asymmetry $(E^{*}_{e^+}-E^{*}_{e^-})/(E^{*}_{e^+}+E^{*}_{e^-})$", ax=axes[0])
closure("E_ep", np.linspace(0, 68, 60), r"$E^{*}_{e^+}$ [MeV]", ax=axes[1])
closure("E_gamma", np.linspace(0, 68, 60), r"$E^{*}_{\gamma}$ [MeV]", ax=axes[2])
fig.suptitle(r"$\pi^0$ Rest Frame")
fig.tight_layout()
savefig("reweight_closure_other")
plt.show()

## Apply to the overlay: lab frame

The weight is a function of the invariants $(m_{ee}, \cos\theta^{*})$, so we apply it to the physically-produced, **boosted** overlay decays (the recovered ~6k sample) using each event's truth $(m_{ee}, \cos\theta^{*})$. The standalone sample has no lab frame ($\pi^0$ at rest), so these panels show the **overlay raw vs reweighted** — how the correction reshapes the daughter kinematics that actually enter the analysis.

In [ ]:
# weight each overlay decay from its truth (m_ee, cos theta*); normalize <w> = 1 over the overlay
w_ov = weight_of(ov_obs["m_ee"], ov_obs["cos_theta_l"])
w_ov *= len(w_ov) / w_ov.sum()

lab_E_gamma = ov["gamma_p4"][:, 0]
lab_E_ep = ov["ep_p4"][:, 0]
lab_opening = np.degrees(np.arccos(cos_between(p3(ov["ep_p4"]), p3(ov["em_p4"]))))
lab_asym = (ov["ep_p4"][:, 0] - ov["em_p4"][:, 0]) / (ov["ep_p4"][:, 0] + ov["em_p4"][:, 0])
lab_ee_gamma_angle = np.degrees(np.arccos(
    cos_between(p3(ov["ep_p4"]) + p3(ov["em_p4"]), p3(ov["gamma_p4"]))))


def raw_vs_rw(ax, var, bins, xlabel):
    ax.hist(var, bins=bins, density=True, histtype="step", lw=1.4,
            color="tab:blue", alpha=0.6, label="BNB NC $\pi^0$ overlay")
    ax.hist(var, bins=bins, density=True, weights=w_ov, histtype="step", lw=2.2,
            color="tab:green", label="BNB NC $\pi^0$ overlay reweighted")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("normalized")
    ax.legend(fontsize=8)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
raw_vs_rw(axes[0, 0], lab_E_gamma, np.linspace(0, np.percentile(lab_E_gamma, 99), 50), r"$E_{\gamma}$ [MeV]")
raw_vs_rw(axes[0, 1], lab_E_ep, np.linspace(0, np.percentile(lab_E_ep, 99), 50), r"$E_{e^+}$ [MeV]")
raw_vs_rw(axes[0, 2], lab_opening, np.linspace(0, np.percentile(lab_opening, 99), 50), r"$e^+e^-$ opening angle [deg]")
raw_vs_rw(axes[1, 0], lab_asym, np.linspace(-1, 1, 50), r"$e^+e^-$ energy asymmetry $(E_{e^+}-E_{e^-})/(E_{e^+}+E_{e^-})$")
raw_vs_rw(axes[1, 1], lab_ee_gamma_angle, np.linspace(0, np.percentile(lab_ee_gamma_angle, 99), 50),
          r"$e^+e^-$ momentum vs $\gamma$ opening angle [deg]")
axes[1, 2].axis("off")
fig.suptitle(r"Lab Frame")
fig.tight_layout()
savefig("reweight_lab_frame")
plt.show()

## Save the per-event overlay weights

Keyed by `run/subrun/event` so they can be joined into the analysis dataframe. A production version would live in `src/` alongside the other reweightings, computing the weight from truth $(m_{ee}, \cos\theta^{*})$ via the same 2D grid.

In [ ]:
out = {
    "run": ov["run"], "subrun": ov["subrun"], "event": ov["event"], "sample": ov["sample"],
    "m_ee": ov_obs["m_ee"], "cos_theta_star": ov_obs["cos_theta_l"],
    "pi0_dalitz_weight": w_ov,
}
pd.DataFrame(out).to_parquet("pi0_dalitz_reweight_demo.parquet")
print(f"saved {len(w_ov):,} overlay weights to pi0_dalitz_reweight_demo.parquet")